# Chapter 10 - Kubernetes Homework

This notebook provides a structured approach to completing the Kubernetes homework.

**Author:** ML Zoomcamp 2025  
**Topic:** Deploying ML Models to Kubernetes  
**Model:** Bank Marketing Prediction

## Configuration Section
### 📝 Modify these parameters as needed

In [ ]:
# ==================== CONFIGURATION ====================
# Modify these variables based on homework requirements

# Docker Configuration
DOCKER_IMAGE_NAME = "zoomcamp-model"
DOCKER_IMAGE_TAG = "3.11.5-hw10"
DOCKER_IMAGE_FULL = f"{DOCKER_IMAGE_NAME}:{DOCKER_IMAGE_TAG}"
DOCKER_HUB_IMAGE = "svizor/zoomcamp-model:3.11.5-hw10"  # Fallback if build fails

# Kubernetes Configuration
CLUSTER_NAME = "ml-zoomcamp-cluster"
DEPLOYMENT_NAME = "bank-marketing-deployment"
SERVICE_NAME = "bank-marketing-service"
HPA_NAME = "bank-marketing-hpa"
NAMESPACE = "default"

# Service Configuration
SERVICE_PORT = 80
TARGET_PORT = 9696
LOCAL_PORT = 9696

# Resource Configuration
MEMORY_REQUEST = "64Mi"
MEMORY_LIMIT = "128Mi"
CPU_REQUEST = "100m"
CPU_LIMIT = "200m"

# HPA Configuration
MIN_REPLICAS = 1
MAX_REPLICAS = 3
CPU_THRESHOLD = 20

# Test Data (Question 1 & 6)
TEST_CLIENT_DATA = {
    "job": "management",
    "duration": 400,
    "poutcome": "success"
}

# Load Test Configuration (Question 8)
LOAD_TEST_REQUESTS = 1000
LOAD_TEST_DELAY = 0.01  # seconds between requests

print("✅ Configuration loaded successfully!")
print(f"Docker Image: {DOCKER_IMAGE_FULL}")
print(f"Deployment: {DEPLOYMENT_NAME}")
print(f"Service: {SERVICE_NAME}")

## Question 1: Local Docker Testing

**Task:** Build and run the Docker container locally, then test the prediction endpoint.

**What to find:** The probability value returned by the model.

In [ ]:
# Check if Docker is running
!docker --version

In [ ]:
# Option 1: Build the Docker image (if you have the Dockerfile)
# Uncomment if you want to build locally
# !docker build -t {DOCKER_IMAGE_FULL} .

# Option 2: Pull the pre-built image
!docker pull {DOCKER_HUB_IMAGE}
!docker tag {DOCKER_HUB_IMAGE} {DOCKER_IMAGE_FULL}

In [ ]:
# Stop any existing container on port 9696
import subprocess
subprocess.run(["docker", "stop", "zoomcamp-test"], capture_output=True)
subprocess.run(["docker", "rm", "zoomcamp-test"], capture_output=True)

# Run the container in detached mode
!docker run -d --name zoomcamp-test -p {LOCAL_PORT}:9696 {DOCKER_IMAGE_FULL}

In [ ]:
# Wait for container to be ready
import time
print("Waiting for container to start...")
time.sleep(3)

# Check container status
!docker ps | grep zoomcamp-test

In [ ]:
# Test the prediction endpoint
import requests
import json

url = f"http://localhost:{LOCAL_PORT}/predict"

try:
    response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
    result = response.json()
    
    print("\n" + "="*50)
    print("🎯 QUESTION 1 ANSWER")
    print("="*50)
    print(f"Input data: {json.dumps(TEST_CLIENT_DATA, indent=2)}")
    print(f"\nPrediction result: {json.dumps(result, indent=2)}")
    
    if 'subscription_probability' in result:
        prob = result['subscription_probability']
        print(f"\n✅ Probability: {prob:.3f}")
        print(f"\nSelect the closest option from the homework choices.")
    else:
        print(f"\n⚠️ Unexpected response format. Full response: {result}")
        
except Exception as e:
    print(f"❌ Error testing endpoint: {e}")
    print("\nCheck if container is running:")
    !docker logs zoomcamp-test --tail 20

In [ ]:
# Clean up - stop the container after testing
# Uncomment when done with local testing
# !docker stop zoomcamp-test
# !docker rm zoomcamp-test

## Question 2: Kind Version

**Task:** Check the installed version of `kind`

In [ ]:
# Check kind version
!kind --version

print("\n" + "="*50)
print("🎯 QUESTION 2 ANSWER")
print("="*50)
print("Record the version number shown above.")

## Question 3: Kubernetes Concepts

**Task:** What is the smallest deployable unit of computing in Kubernetes?

**Options:**
- Node
- Pod ✅ (Expected answer)
- Deployment
- Service

In [ ]:
print("="*50)
print("🎯 QUESTION 3 ANSWER")
print("="*50)
print("\nThe smallest deployable unit in Kubernetes is: Pod")
print("\nExplanation:")
print("- A Pod is a group of one or more containers")
print("- Pods are the atomic unit of scheduling in Kubernetes")
print("- Deployments and Services manage Pods")
print("- Nodes are physical/virtual machines that run Pods")

## Kubernetes Cluster Setup

In [ ]:
# Check if kubectl is installed
!kubectl version --client

In [ ]:
# Delete existing cluster if it exists
!kind delete cluster --name {CLUSTER_NAME} 2>/dev/null || true

In [ ]:
# Create a new kind cluster
!kind create cluster --name {CLUSTER_NAME}

print("\nWaiting for cluster to be ready...")
!kubectl cluster-info --context kind-{CLUSTER_NAME}

## Question 4: Service Types

**Task:** What service type is running after cluster creation?

In [ ]:
# List all services
!kubectl get services --all-namespaces

print("\n" + "="*50)
print("🎯 QUESTION 4 ANSWER")
print("="*50)
print("Look at the TYPE column for the kubernetes service above.")
print("Expected answer: ClusterIP")

## Question 5: Load Docker Image to Kind

**Task:** What command loads a Docker image into the kind cluster?

In [ ]:
print("="*50)
print("🎯 QUESTION 5 ANSWER")
print("="*50)
print("\nCommand to load Docker image into kind:")
print(f"kind load docker-image {DOCKER_IMAGE_FULL} --name {CLUSTER_NAME}")
print("\nExecuting the command...\n")

# Actually load the image
!kind load docker-image {DOCKER_IMAGE_FULL} --name {CLUSTER_NAME}

## Question 6: Create Deployment

**Task:** Create a deployment with the specified configuration

In [ ]:
# Apply the deployment from the YAML file
!kubectl apply -f deployment.yaml

print("\nWaiting for deployment to be ready...")
!kubectl wait --for=condition=available --timeout=60s deployment/{DEPLOYMENT_NAME}

print("\nDeployment status:")
!kubectl get deployments
!kubectl get pods

In [ ]:
# Describe the deployment to verify configuration
!kubectl describe deployment {DEPLOYMENT_NAME}

print("\n" + "="*50)
print("🎯 QUESTION 6 ANSWER")
print("="*50)
print("Check the deployment.yaml file for the containerPort value.")
print(f"Expected containerPort: {TARGET_PORT}")

## Question 7: Create LoadBalancer Service

**Task:** Create a LoadBalancer service and identify the selector value

In [ ]:
# Apply the service from the YAML file
!kubectl apply -f service.yaml

print("\nService created:")
!kubectl get services

print("\nService details:")
!kubectl describe service {SERVICE_NAME}

In [ ]:
print("="*50)
print("🎯 QUESTION 7 ANSWER")
print("="*50)
print("Check the service.yaml file for the selector app value.")
print("The selector should match the label in the deployment.")
print(f"\nExpected selector: app: bank-marketing")

## Test the Service with Port Forwarding

In [ ]:
# Note: Port forwarding needs to run in background
# You may need to run this in a separate terminal:
print(f"Run this command in a separate terminal:")
print(f"kubectl port-forward service/{SERVICE_NAME} {LOCAL_PORT}:{SERVICE_PORT}")
print("\nThen test with the cell below.")

In [ ]:
# Test the service through port-forward
# Make sure port-forward is running before executing this cell

import requests
import json

url = f"http://localhost:{LOCAL_PORT}/predict"

try:
    response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
    result = response.json()
    
    print("\n✅ Service is responding!")
    print(f"Result: {json.dumps(result, indent=2)}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure port-forward is running!")

## Question 8: Horizontal Pod Autoscaler (Optional)

**Task:** Configure HPA and perform load testing to see maximum replicas

In [ ]:
# Install metrics-server for HPA (required for kind)
print("Installing metrics-server...")
!kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml

# Patch metrics-server for kind (insecure TLS)
!kubectl patch deployment metrics-server -n kube-system --type='json' -p='[{"op": "add", "path": "/spec/template/spec/containers/0/args/-", "value": "--kubelet-insecure-tls"}]'

print("\nWaiting for metrics-server to be ready...")
import time
time.sleep(10)
!kubectl wait --for=condition=available --timeout=60s deployment/metrics-server -n kube-system

In [ ]:
# Apply HPA configuration
!kubectl apply -f hpa.yaml

print("\nHPA status:")
!kubectl get hpa

In [ ]:
# Run load test
print(f"Starting load test with {LOAD_TEST_REQUESTS} requests...")
print("This will take a few minutes.\n")

import requests
import time

url = f"http://localhost:{LOCAL_PORT}/predict"
successful_requests = 0
failed_requests = 0

for i in range(LOAD_TEST_REQUESTS):
    try:
        response = requests.post(url, json=TEST_CLIENT_DATA, timeout=5)
        if response.status_code == 200:
            successful_requests += 1
        else:
            failed_requests += 1
    except Exception as e:
        failed_requests += 1
    
    if i % 100 == 0:
        print(f"Progress: {i}/{LOAD_TEST_REQUESTS} requests sent")
    
    time.sleep(LOAD_TEST_DELAY)

print(f"\n✅ Load test complete!")
print(f"Successful: {successful_requests}")
print(f"Failed: {failed_requests}")

In [ ]:
# Monitor HPA and pods during/after load test
print("HPA Status:")
!kubectl get hpa

print("\nPod Status:")
!kubectl get pods

print("\nDeployment Replicas:")
!kubectl get deployment {DEPLOYMENT_NAME}

print("\n" + "="*50)
print("🎯 QUESTION 8 ANSWER")
print("="*50)
print("Count the number of READY pods above.")
print("This is the maximum number of replicas achieved during load test.")

In [ ]:
# Watch HPA over time to see scaling behavior
# Note: This will run for 2 minutes
print("Monitoring HPA for 2 minutes...\n")

import time
for i in range(24):  # 24 * 5 seconds = 2 minutes
    !kubectl get hpa {HPA_NAME}
    !kubectl get pods | grep {DEPLOYMENT_NAME}
    print(f"\nTime: {i*5} seconds\n")
    time.sleep(5)

## Cleanup

Run these cells when you're done to clean up resources

In [ ]:
# Delete Kubernetes resources
!kubectl delete hpa {HPA_NAME}
!kubectl delete service {SERVICE_NAME}
!kubectl delete deployment {DEPLOYMENT_NAME}

In [ ]:
# Delete the kind cluster
!kind delete cluster --name {CLUSTER_NAME}

In [ ]:
# Stop and remove Docker container
!docker stop zoomcamp-test 2>/dev/null || true
!docker rm zoomcamp-test 2>/dev/null || true

## Summary of Answers

Run this cell to see all answers in one place:

In [ ]:
print("="*60)
print("     CHAPTER 10 KUBERNETES HOMEWORK - ANSWER SUMMARY")
print("="*60)
print("\nQ1: Model Probability")
print("    → Run the local Docker test to get the probability value")
print("\nQ2: Kind Version")
print("    → Run 'kind --version' to get the version")
print("\nQ3: Smallest Kubernetes Unit")
print("    → Answer: Pod")
print("\nQ4: Default Service Type")
print("    → Answer: ClusterIP")
print("\nQ5: Load Image to Kind")
print("    → Command: kind load docker-image")
print("\nQ6: Container Port")
print("    → Check deployment.yaml for containerPort value")
print("\nQ7: Service Selector")
print("    → Check service.yaml for selector app value")
print("\nQ8: Maximum Replicas (Optional)")
print("    → Count pods after load test")
print("\n" + "="*60)